In [10]:
import torch
import torchaudio
import json
from einops import rearrange
from stable_audio_tools.models.factory import create_model_from_config
from stable_audio_tools.models.utils import load_ckpt_state_dict
from stable_audio_tools.inference.generation import generate_diffusion_cond

In [7]:
CONFIG_PATH = "checkpoints/model_config_hybrid.json"
CKPT_PATH = "runs/hybrid_v1/models/epoch=22-step=6000.ckpt" 
OUTPUT_FILENAME = "generations/generated_test.wav"

TARGET_SPECIES_ID = 4    
TARGET_STR = "Warbling Vireo"
CFG_SCALE = 6.0           
STEPS = 100               
AUDIO_LENGTH_SECONDS = 15

In [4]:
with open(CONFIG_PATH, "r") as f:
    model_config = json.load(f)

model = create_model_from_config(model_config)
raw_state_dict = load_ckpt_state_dict(CKPT_PATH)

clean_state_dict = {}
for key, value in raw_state_dict.items():
    if key.startswith('diffusion_ema.ema_model.'):
        new_key = key.replace('diffusion_ema.ema_model.', '')
        clean_state_dict[new_key] = value
        
    elif key.startswith('diffusion.'):
        new_key = key.replace('diffusion.', '', 1)
        if new_key not in clean_state_dict:
            clean_state_dict[new_key] = value

model.load_state_dict(clean_state_dict, strict=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device, dtype=torch.float16)
model.eval()
print(device)

C:\Users\Leonard Leber\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading weights:   0%|          | 0/99 [00:00<?, ?it/s]

ConditionedDiffusionModelWrapper(
  (model): DiTWrapper(
    (model): DiffusionTransformer(
      (timestep_features): FourierFeatures()
      (to_timestep_embed): Sequential(
        (0): Linear(in_features=256, out_features=1536, bias=True)
        (1): SiLU()
        (2): Linear(in_features=1536, out_features=1536, bias=True)
      )
      (to_cond_embed): Sequential(
        (0): Linear(in_features=768, out_features=768, bias=False)
        (1): SiLU()
        (2): Linear(in_features=768, out_features=768, bias=False)
      )
      (to_global_embed): Sequential(
        (0): Linear(in_features=2304, out_features=1536, bias=False)
        (1): SiLU()
        (2): Linear(in_features=1536, out_features=1536, bias=False)
      )
      (transformer): ContinuousTransformer(
        (layers): ModuleList(
          (0-23): 24 x TransformerBlock(
            (pre_norm): LayerNorm()
            (self_attn): Attention(
              (to_qkv): Linear(in_features=1536, out_features=4608, bias=F

In [8]:
sample_rate = model_config["sample_rate"]
sample_size = int(AUDIO_LENGTH_SECONDS * sample_rate)

conditioning = [{
    "prompt": f"A field recording of a {TARGET_STR} singing in nature, stereo audio.",
    "seconds_start": 0,
    "seconds_total": AUDIO_LENGTH_SECONDS,
    "species_id": TARGET_SPECIES_ID
}]

In [11]:
with torch.no_grad():
    output = generate_diffusion_cond(
        model,
        steps=STEPS,
        cfg_scale=CFG_SCALE,
        conditioning=conditioning,
        sample_size=sample_size,
        sigma_min=0.3,
        sigma_max=500,
        sampler_type="dpmpp-3m-sde",
        device=device,
        seed=14
    )

14


C:\Users\Leonard Leber\AppData\Local\Programs\Python\Python311\Lib\site-packages\stable_audio_tools\models\conditioners.py:362: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16) and torch.set_grad_enabled(self.enable_grad):


  0%|          | 0/100 [00:00<?, ?it/s]

C:\Users\Leonard Leber\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchsde\_brownian\brownian_interval.py:608: UserWarning: Should have tb<=t1 but got tb=500.00006103515625 and t1=500.000061.
  warnings.warn(f"Should have {tb_name}<=t1 but got {tb_name}={tb} and t1={self._end}.")


In [14]:
output = output.to(torch.float32).div(torch.max(torch.abs(output))).clamp(-1, 1).cpu()
torchaudio.save(OUTPUT_FILENAME, output, sample_rate)